# Build the nested ablation cache (GPU)

This is the only leg that needs an accelerator. It reads the Provo corpus and a
reference language model and writes a cache of `K + 1` next-word distributions per
target, one for every truncation depth. Everything downstream reads that cache and
runs on CPU.

Session settings: **GPU T4 x2**, internet **on**.

Attach two datasets before running:

1. Provo, holding `Provo_Corpus-Predictability_Norms.csv` and
   `Provo_Corpus-Eyetracking_Data.csv` from <https://osf.io/sjefs/>.
2. SUBTLEX-US, holding `SUBTLEXusfrequencyabove1.csv` from
   <https://www.ugent.be/pp/experimentele-psychologie/en/research/documents/subtlexus>.

Set `PROVO_DIR` and `SUBTLEX` below to wherever Kaggle mounted them, then run the
notebook top to bottom. Save the output directory as a Kaggle dataset when it
finishes, and point the estimation notebook at it.

In [ ]:
!pip -q install 'transformers>=4.40' 'accelerate>=0.30' pyyaml

In [ ]:
!git clone -q https://github.com/garyzhang1006/lossy-context.git /kaggle/working/lossy-context
!pip -q install -e /kaggle/working/lossy-context

## Where the data is

Edit these two paths to match the datasets you attached. The cell fails loudly if
either is missing, which is cheaper than discovering it after the model has loaded.

In [ ]:
import os, glob, sys
from pathlib import Path

PROVO_DIR = '/kaggle/input/provo-corpus'
SUBTLEX   = '/kaggle/input/subtlex-us/SUBTLEXusfrequencyabove1.csv'
OUT       = '/kaggle/working/build'
MODEL     = 'Qwen/Qwen2.5-1.5B'

for p in (PROVO_DIR, SUBTLEX):
    if not os.path.exists(p):
        raise SystemExit(f'not found: {p}\navailable: ' + str(glob.glob('/kaggle/input/*')))
print(sorted(os.listdir(PROVO_DIR)))

## Check the install before spending GPU time

The self-test builds a synthetic corpus in memory and runs all four experiments on
it. It touches no data and no accelerator, and it takes about a minute. If it does
not print `SELFTEST PASSED`, stop here.

In [ ]:
!lcsa selftest --out /kaggle/working/selftest

## Confirm the accelerator

A T4 pair reports two devices and compute capability 7.5. If this cell reports a
CPU, the session was started without the accelerator and the build will take days
instead of hours.

In [ ]:
import torch
print('cuda:', torch.cuda.is_available(), 'devices:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i))

## A small build first

Twenty targets take a couple of minutes and exercise every code path the full run
uses: the encoding fallback on the norms file, the candidate set, the packed forward
pass and the cache gate. A failure here costs minutes; the same failure four hours
into the full build costs the session.

In [ ]:
# argparse does not expand Python variables, so build the command explicitly.
import subprocess

def run(cmd):
    print(' '.join(cmd))
    r = subprocess.run(cmd)
    if r.returncode != 0:
        raise SystemExit(f'command failed with status {r.returncode}')

run(['lcsa', '--verbose', 'build',
     '--provo-dir', PROVO_DIR, '--subtlex', SUBTLEX, '--model', MODEL,
     '--out', '/kaggle/working/smoke', '--limit', '20'])

In [ ]:
import json
print(json.dumps(json.load(open('/kaggle/working/smoke/g0.json')), indent=2)[:1200])
print(json.dumps(json.load(open('/kaggle/working/smoke/g0_cache.json')), indent=2))
print(json.dumps(json.load(open('/kaggle/working/smoke/g1.json')), indent=2))

## Estimate the full build from the smoke run

The throughput above is measured on the same code path the full build uses, so it
extrapolates honestly. Provo has 2,687 targets and a mean context of 21.86 words,
giving 61,435 distinct context forward passes. Kaggle stops a GPU session at twelve
hours, so if the estimate exceeds about ten, lower `--max-depth` or split the build
across two sessions with `--limit` and merge the caches.

In [ ]:
g1 = json.load(open('/kaggle/working/smoke/g1.json'))
rate = g1['contexts_per_second']
print(f"{rate:.1f} contexts/s on the smoke run")
print(f"full build estimate: {61435 / rate / 3600:.2f} hours")

## The full build

This writes `cache.npz`, `targets.csv`, `candidates.json` and the gate records to
`OUT`. It logs every fifty targets, so a stalled run is visible in the output rather
than only in the wall clock.

In [ ]:
run(['lcsa', '--verbose', 'build',
     '--provo-dir', PROVO_DIR, '--subtlex', SUBTLEX, '--model', MODEL,
     '--out', OUT, '--max-depth', '32', '--max-candidates', '120', '--top-k', '50'])

## What to keep

`cache.npz` is the expensive object and holds plain arrays with nothing pickled, so
it loads on any machine. `targets.csv` carries the `(text_id, word_number)` key of
every built target in cache order and is what joins the eye-tracking arm back on.
`candidates.json` carries the frozen candidate word list per target, which the topic
and order nulls need. Save all four to a dataset and attach it to the estimation
notebook.

In [ ]:
for p in sorted(Path(OUT).iterdir()):
    print(f'{p.name:24s} {p.stat().st_size / 1e6:8.2f} MB')